<a href="https://colab.research.google.com/github/leogiarola/I2A2/blob/main/RAG_I2A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Download dos documentos em PDF
import os
if not os.path.exists("dados_rag_multiformato"):
    os.makedirs("dados_rag_multiformato")
# The URL containing the CSV data
url = 'https://portaldatransparencia.gov.br/notas-fiscais/consulta/baixar?paginacaoSimples=true&direcaoOrdenacao=asc&de=01%2F06%2F2025&ate=02%2F06%2F2025&colunasSelecionadas=linkDetalhamento%2CorgaoSuperiorDestinatario%2CorgaoDestinatario%2CnomeFornecedor%2CcnpjFornecedor%2CmunicipioFornecedor%2CufFornecedor%2CchaveNotaFiscal%2CvalorNotaFiscal%2CdataEmissao%2CtipoEventoMaisRecente%2Cnumero%2Cserie%2CcnpjOrgaoDestinatario%2CdataTipoEventoMaisRecente'
#!wget https://www.dropbox.com/scl/fi/hn7zzvhj362w91io2c2pc/edital-inova-cemig-desafio-de-pdi-20.pdf?rlkey=9ghqqre854cshv2el6nxug4ji -O dados_rag_multiformato/edital_inova.pdf
#!wget 'https://www.dropbox.com/scl/fi/w7dvid9egc1zywf3jjgvm/cemig-apresenta-ao2022.pptx?rlkey=7t5efwz1d9pliq18wpblspcwc&st=7dhmhqk6&dl' -O dados_rag_multiformato/cemig_apresentacao.pptx
!wget "{url}" -O dados_rag_multiformato/dados_notas_fiscais.csv

In [ ]:
# Instalar as bibliotecas que usaremos
!pip install -q pdfplumber langchain langchain_community sentence-transformers faiss-cpu python-pptx pandas langchain-openai langchain-huggingface transformers torch accelerate ragas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7

In [ ]:
from google.colab import userdata
OPENAI_KEY = userdata.get('OPENAI_API_KEY')

In [ ]:
import pandas as pd
import pdfplumber
from pptx import Presentation

#Funções para leitura de diferentes tipos de arquivo
def read_txt(path):
    #Lê o conteúdo de um arquivo .txt
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"Erro ao ler o arquivo TXT {path}: {e}")
        return ""

def read_pdf(path):
    #Extrai o texto de um arquivo .pdf
    try:
        with pdfplumber.open(path) as pdf:
            texto = ""
            for pagina in pdf.pages:
                texto += pagina.extract_text() + "\n"
        return texto
    except Exception as e:
        print(f"Erro ao ler o arquivo PDF {path}: {e}")
        return ""

def read_pptx(path):
    #Extrai o texto de uma apresentação .pptx
    try:
        apresentacao = Presentation(path)
        texto = ""
        for slide in apresentacao.slides:
            for shape in slide.shapes:
                if hasattr(shape, "text"):
                    texto += shape.text + "\n"
        return texto
    except Exception as e:
        print(f"Erro ao ler o arquivo PPTX {path}: {e}")
        return ""

def read_csv(path):
    #Lê um arquivo .csv e o converte para uma string formatada
    try:
        df = pd.read_csv(path, sep=';')
        return df.to_string()
    except Exception as e:
        print(f"Erro ao ler o arquivo CSV {path}: {e}")
        return ""

def carregar_documentos_do_diretorio(caminho_diretorio):
    documentos = []
    print(f"Lendo arquivos do diretório: {caminho_diretorio}")
    for nome_arquivo in os.listdir(caminho_diretorio):
        caminho_completo = os.path.join(caminho_diretorio, nome_arquivo)
        conteudo = ""
        if nome_arquivo.endswith(".txt"):
            conteudo = read_txt(caminho_completo)
        elif nome_arquivo.endswith(".pdf"):
            conteudo = read_pdf(caminho_completo)
        elif nome_arquivo.endswith(".pptx"):
            conteudo = read_pptx(caminho_completo)
        elif nome_arquivo.endswith(".csv"):
            conteudo = read_csv(caminho_completo)

        if conteudo:
            documento = Document(
                page_content=conteudo,
                metadata={"fonte": nome_arquivo}
            )
            documentos.append(documento)
            print(f" - Arquivo '{nome_arquivo}' carregado com sucesso.")
    return documentos

In [ ]:
from langchain_core.documents import Document
import logging

logging.getLogger("pdfminer").setLevel(logging.ERROR)

#Carrega todos os documentos da pasta
documentos_carregados = carregar_documentos_do_diretorio("dados_rag_multiformato")
print(f"\nTotal de documentos processados: {len(documentos_carregados)}")

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

#Divide os documentos em chunks que serão processados pelo retriever
print("\nDividindo os documentos em chunks...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
documentos_divididos = text_splitter.split_documents(documentos_carregados)
print(f"Total de chunks criados: {len(documentos_divididos)}")

dados_para_df = [
    {
        'fonte': doc.metadata['fonte'],
        'conteudo': doc.page_content,
    }
    for doc in documentos_divididos
]
df = pd.DataFrame(dados_para_df)
display(df)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

#Cria os embeddings e o Vector Store
print("\nGerando embeddings e criando o Vector Store...")
embedding_function = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")
db = FAISS.from_documents(documentos_divididos, embedding_function)
print("Vector Store criado com sucesso.")

#Define o template do prompt para o LLM
template_prompt = """
### Instruções:
Você é um assistente pessoal responsável por responder perguntas dos usuários, com base no contexto fornecido.
Sempre que possível, cite a fonte da sua informação no final da resposta, como por exemplo: (Fonte: nome_do_arquivo).

### Documentos:
{context}

### Pergunta:
{query}

### Resposta:
"""
prompt = PromptTemplate(
    input_variables=["context", "query"],
    template=template_prompt,
)

In [ ]:
#Importando e configurando o LLM
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_openai import ChatOpenAI
import os

os.environ["OPENAI_API_KEY"] = OPENAI_KEY

try:
    llm_gpt = ChatOpenAI(model_name="gpt-4o-mini")
    print("LLM configurado com sucesso.")
except Exception as e:
    print(f"\nErro ao configurar o LLM: {e}")
    llm_gpt = None

In [ ]:
def formatar_documentos(docs):
    return "\n\n".join(f"Fonte: {doc.metadata['fonte']}\nConteúdo: {doc.page_content}" for doc in docs)


#Configura retriever com o número de chunks a serem recuperados
retriever = db.as_retriever(search_kwargs={'k': 5})

#Cria a chain de execução
rag_gpt = (
    {"context": retriever | formatar_documentos, "query": RunnablePassthrough()}
    | prompt
    | llm_gpt
    | StrOutputParser()
)

In [ ]:
#Testando bot
pergunta = "do que se trata meus dados?"
print(f"\n--- Executando a pergunta: '{pergunta}' ---")

#Recuperando o contexto relevante
documentos_contexto = retriever.invoke(pergunta)
print("\n--- Contexto Recuperado para a Pergunta ---")
print(formatar_documentos(documentos_contexto))
print("-------------------------------------------\n")

print("\n>>> Resposta do modelo:")
resultado_gpt = rag_gpt.invoke(pergunta)

print(resultado_gpt)